## Weather JSON Inspection

Inspect the actual JSON structure stored in the Bronze Weather table.

In [0]:
-- Check the Bronze Weather table structure and content
SELECT 
  source_system,
  source_url,
  source_file,
  batch_id,
  ingested_at,
  LENGTH(raw_json) AS json_length_bytes
FROM `ftw-week-08`.`01_bronze`.`weather_raw`

In [0]:
-- Inspect the JSON structure - look at the top-level keys and hourly data structure
WITH parsed AS (
  SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
)
SELECT 
  json_data.latitude AS latitude,
  json_data.longitude AS longitude,
  json_data.timezone AS timezone,
  json_data.timezone_abbreviation AS timezone_abbreviation,
  SIZE(json_data.hourly.time) AS hourly_time_count,
  SIZE(json_data.hourly.temperature_2m) AS hourly_temperature_count,
  SIZE(json_data.hourly.precipitation) AS hourly_precipitation_count,
  SIZE(json_data.hourly.rain) AS hourly_rain_count,
  SIZE(json_data.hourly.snowfall) AS hourly_snowfall_count,
  SIZE(json_data.hourly.weather_code) AS hourly_weather_code_count,
  SIZE(json_data.hourly.wind_speed_10m) AS hourly_wind_speed_count
FROM parsed

In [0]:
-- Sample first and last few hourly timestamps from the JSON payload
WITH parsed AS (
  SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
)
SELECT 
  json_data.hourly.time[0] AS first_timestamp,
  json_data.hourly.time[1] AS second_timestamp,
  json_data.hourly.time[2] AS third_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 3] AS third_to_last_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 2] AS second_to_last_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 1] AS last_timestamp
FROM parsed

## Weather Silver Transformation

Create the Silver Weather table by parsing and flattening the JSON hourly arrays into one row per hourly timestamp.

In [0]:
-- Validate that all hourly arrays have the same length before POSEXPLODE
WITH parsed AS (
  SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
),
array_sizes AS (
  SELECT
    CASE WHEN json_data IS NOT NULL THEN 'SUCCESS' ELSE 'FAILED' END AS json_parse_status,
    SIZE(json_data.hourly.time) AS time_count,
    SIZE(json_data.hourly.temperature_2m) AS temperature_2m_count,
    SIZE(json_data.hourly.precipitation) AS precipitation_count,
    SIZE(json_data.hourly.rain) AS rain_count,
    SIZE(json_data.hourly.snowfall) AS snowfall_count,
    SIZE(json_data.hourly.weather_code) AS weather_code_count,
    SIZE(json_data.hourly.wind_speed_10m) AS wind_speed_10m_count
  FROM parsed
)
SELECT
  json_parse_status,
  time_count,
  temperature_2m_count,
  precipitation_count,
  rain_count,
  snowfall_count,
  weather_code_count,
  wind_speed_10m_count,
  CASE
    WHEN json_parse_status = 'FAILED' THEN 'FAIL'
    WHEN time_count = 0 THEN 'FAIL'
    WHEN time_count = temperature_2m_count
     AND time_count = precipitation_count
     AND time_count = rain_count
     AND time_count = snowfall_count
     AND time_count = weather_code_count
     AND time_count = wind_speed_10m_count
    THEN 'PASS'
    ELSE 'FAIL'
  END AS array_alignment_status
FROM array_sizes

In [0]:
-- Create Silver Weather table schema with TIMESTAMP_NTZ support
-- TIMESTAMP_NTZ preserves local-hour labels without session timezone conversion
DROP TABLE IF EXISTS `ftw-week-08`.`02_silver`.`weather`;

CREATE TABLE `ftw-week-08`.`02_silver`.`weather` (
  weather_datetime TIMESTAMP_NTZ,
  latitude DOUBLE,
  longitude DOUBLE,
  timezone STRING,
  timezone_abbreviation STRING,
  utc_offset_seconds INT,
  temperature_2m DOUBLE,
  precipitation DOUBLE,
  rain DOUBLE,
  snowfall DOUBLE,
  weather_code INT,
  wind_speed_10m DOUBLE,
  source_system STRING,
  source_url STRING,
  source_file STRING,
  batch_id STRING,
  ingested_at TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported')

In [0]:
-- Insert Weather data by exploding and flattening the hourly JSON arrays
INSERT INTO `ftw-week-08`.`02_silver`.`weather`
WITH parsed_json AS (
  SELECT 
    from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data,
    source_system,
    source_url,
    source_file,
    batch_id,
    ingested_at
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
),
exploded_weather AS (
  SELECT 
    -- Explode the hourly time array with position to maintain alignment
    POSEXPLODE(json_data.hourly.time) AS (hour_index, hourly_time_str),
    json_data,
    source_system,
    source_url,
    source_file,
    batch_id,
    ingested_at
  FROM parsed_json
)
SELECT
  -- Convert hourly timestamp string to TIMESTAMP_NTZ type (preserves NYC local-hour labels without session timezone conversion)
  TRY_CAST(hourly_time_str AS TIMESTAMP_NTZ) AS weather_datetime,
  
  -- Source location and timezone metadata
  json_data.latitude AS latitude,
  json_data.longitude AS longitude,
  json_data.timezone AS timezone,
  json_data.timezone_abbreviation AS timezone_abbreviation,
  json_data.utc_offset_seconds AS utc_offset_seconds,
  
  -- Extract weather measurements by position using hour_index to maintain alignment
  json_data.hourly.temperature_2m[hour_index] AS temperature_2m,
  json_data.hourly.precipitation[hour_index] AS precipitation,
  json_data.hourly.rain[hour_index] AS rain,
  json_data.hourly.snowfall[hour_index] AS snowfall,
  json_data.hourly.weather_code[hour_index] AS weather_code,
  json_data.hourly.wind_speed_10m[hour_index] AS wind_speed_10m,
  
  -- Bronze provenance columns
  source_system,
  source_url,
  source_file,
  batch_id,
  ingested_at
FROM exploded_weather
ORDER BY weather_datetime

## Weather Silver Validation

Validate the Silver Weather table structure and data quality.

In [0]:
-- Validate schema and data types
DESCRIBE `ftw-week-08`.`02_silver`.`weather`

In [0]:
-- Basic row count and timestamp range
SELECT
  COUNT(*) AS total_row_count,
  MIN(weather_datetime) AS min_weather_datetime,
  MAX(weather_datetime) AS max_weather_datetime,
  DATEDIFF(DAY, MIN(weather_datetime), MAX(weather_datetime)) + 1 AS days_covered
FROM `ftw-week-08`.`02_silver`.`weather`

In [0]:
-- Check for duplicate timestamps
SELECT
  weather_datetime,
  COUNT(*) AS occurrence_count
FROM `ftw-week-08`.`02_silver`.`weather`
GROUP BY weather_datetime
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC, weather_datetime

In [0]:
-- NULL count for every weather field
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN weather_datetime IS NULL THEN 1 ELSE 0 END) AS null_weather_datetime,
  SUM(CASE WHEN temperature_2m IS NULL THEN 1 ELSE 0 END) AS null_temperature_2m,
  SUM(CASE WHEN precipitation IS NULL THEN 1 ELSE 0 END) AS null_precipitation,
  SUM(CASE WHEN rain IS NULL THEN 1 ELSE 0 END) AS null_rain,
  SUM(CASE WHEN snowfall IS NULL THEN 1 ELSE 0 END) AS null_snowfall,
  SUM(CASE WHEN weather_code IS NULL THEN 1 ELSE 0 END) AS null_weather_code,
  SUM(CASE WHEN wind_speed_10m IS NULL THEN 1 ELSE 0 END) AS null_wind_speed_10m
FROM `ftw-week-08`.`02_silver`.`weather`

In [0]:
-- Min/Max profiling for numeric weather fields (investigation only)
SELECT
  MIN(temperature_2m) AS min_temperature_2m,
  MAX(temperature_2m) AS max_temperature_2m,
  MIN(precipitation) AS min_precipitation,
  MAX(precipitation) AS max_precipitation,
  MIN(rain) AS min_rain,
  MAX(rain) AS max_rain,
  MIN(snowfall) AS min_snowfall,
  MAX(snowfall) AS max_snowfall,
  MIN(wind_speed_10m) AS min_wind_speed_10m,
  MAX(wind_speed_10m) AS max_wind_speed_10m
FROM `ftw-week-08`.`02_silver`.`weather`

In [0]:
-- Check for negative values in precipitation, rain, snowfall, and wind_speed
SELECT
  COUNT_IF(precipitation < 0) AS negative_precipitation_count,
  COUNT_IF(rain < 0) AS negative_rain_count,
  COUNT_IF(snowfall < 0) AS negative_snowfall_count,
  COUNT_IF(wind_speed_10m < 0) AS negative_wind_speed_count,

  MIN(precipitation) AS min_precipitation,
  MIN(rain) AS min_rain,
  MIN(snowfall) AS min_snowfall,
  MIN(wind_speed_10m) AS min_wind_speed_10m
FROM `ftw-week-08`.`02_silver`.`weather`;

In [0]:
-- Investigate unusual values such as -999 across numeric weather fields (investigation only)
SELECT
  weather_datetime,
  temperature_2m,
  precipitation,
  rain,
  snowfall,
  weather_code,
  wind_speed_10m
FROM `ftw-week-08`.`02_silver`.`weather`
WHERE temperature_2m = -999
   OR precipitation = -999
   OR rain = -999
   OR snowfall = -999
   OR wind_speed_10m = -999
ORDER BY weather_datetime

In [0]:
-- Check for timestamps outside the requested period (2026-03-01 through 2026-05-31)
SELECT
  COUNT(*) AS timestamps_outside_period,
  MIN(weather_datetime) AS earliest_outside,
  MAX(weather_datetime) AS latest_outside
FROM `ftw-week-08`.`02_silver`.`weather`
WHERE weather_datetime < '2026-03-01 00:00:00'
   OR weather_datetime > '2026-05-31 23:59:59'

In [0]:
-- Validate that all weather_code values are documented Open-Meteo WMO codes
WITH valid_codes AS (
  SELECT code FROM VALUES
    (0), (1), (2), (3),
    (45), (48),
    (51), (53), (55), (56), (57),
    (61), (63), (65), (66), (67),
    (71), (73), (75), (77),
    (80), (81), (82), (85), (86),
    (95), (96), (99)
  AS valid_codes(code)
)
SELECT
  weather_code,
  COUNT(*) AS occurrence_count
FROM `ftw-week-08`.`02_silver`.`weather`
WHERE weather_code IS NOT NULL
  AND weather_code NOT IN (SELECT code FROM valid_codes)
GROUP BY weather_code
ORDER BY occurrence_count DESC, weather_code

## Weather Coverage and Data Quality

Detect missing hourly timestamps and compare actual vs. expected observations.

In [0]:
-- Detect missing hourly timestamps by comparing each timestamp with the next
WITH ordered_timestamps AS (
  SELECT 
    weather_datetime,
    LEAD(weather_datetime) OVER (ORDER BY weather_datetime) AS next_datetime,
    TIMESTAMPDIFF(HOUR, weather_datetime, LEAD(weather_datetime) OVER (ORDER BY weather_datetime)) AS hours_gap
  FROM `ftw-week-08`.`02_silver`.`weather`
)
SELECT
  weather_datetime AS timestamp_before_gap,
  next_datetime AS timestamp_after_gap,
  hours_gap,
  hours_gap - 1 AS missing_hours
FROM ordered_timestamps
WHERE hours_gap > 1
ORDER BY weather_datetime

In [0]:
-- Inspect source-provided local-hour labels on DST transition day (March 8, 2026)
-- INSPECTION ONLY: This query shows what local-hour labels the source provided.
-- DST begins on March 8, 2026 at 2:00 AM (clocks spring forward to 3:00 AM).
-- The source may provide 24 labels (00:00-23:00) or 23 labels (skipping 02:00).
-- This is an observation of the source data, not proof of DST correctness.
SELECT weather_datetime
FROM `ftw-week-08`.`02_silver`.`weather`
WHERE CAST(weather_datetime AS DATE) = DATE '2026-03-08'
ORDER BY weather_datetime

In [0]:
-- Compare the number of timestamps in the approved Bronze batch vs. Silver row count for the same batch
WITH bronze_count AS (
  SELECT 
    source_file,
    batch_id,
    SIZE(from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>').hourly.time) AS source_hour_count
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
),
silver_count AS (
  SELECT 
    source_file,
    batch_id,
    COUNT(*) AS silver_hour_count
  FROM `ftw-week-08`.`02_silver`.`weather`
  WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
    AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
  GROUP BY source_file, batch_id
)
SELECT
  b.source_file,
  b.batch_id,
  b.source_hour_count,
  COALESCE(s.silver_hour_count, 0) AS silver_hour_count,
  b.source_hour_count - COALESCE(s.silver_hour_count, 0) AS difference
FROM bronze_count b
LEFT JOIN silver_count s ON b.source_file = s.source_file AND b.batch_id = s.batch_id

In [0]:
-- Expected vs actual observations summary
WITH stats AS (
  SELECT
    COUNT(*) AS actual_observations,
    MIN(weather_datetime) AS min_datetime,
    MAX(weather_datetime) AS max_datetime,
    TIMESTAMPDIFF(HOUR, MIN(weather_datetime), MAX(weather_datetime)) + 1 AS expected_hourly_observations
  FROM `ftw-week-08`.`02_silver`.`weather`
)
SELECT
  actual_observations,
  expected_hourly_observations,
  expected_hourly_observations - actual_observations AS missing_observations,
  ROUND(100.0 * actual_observations / expected_hourly_observations, 2) AS coverage_percentage,
  min_datetime,
  max_datetime
FROM stats

In [0]:
-- Sample first 10 weather records
SELECT *
FROM `ftw-week-08`.`02_silver`.`weather`
ORDER BY weather_datetime
LIMIT 10

In [0]:
-- Sample last 10 weather records
SELECT *
FROM `ftw-week-08`.`02_silver`.`weather`
ORDER BY weather_datetime DESC
LIMIT 10

In [0]:
-- Final validation summary with Pass/Fail status
WITH validation_results AS (
    SELECT
        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS total_rows,

        (SELECT TIMESTAMPDIFF(HOUR, MIN(weather_datetime), MAX(weather_datetime)) + 1
         FROM `ftw-week-08`.`02_silver`.`weather`) AS expected_observations,

        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS actual_observations,

        (SELECT COUNT(*)
         FROM (
             SELECT weather_datetime, COUNT(*) AS cnt
             FROM `ftw-week-08`.`02_silver`.`weather`
             GROUP BY weather_datetime
             HAVING COUNT(*) > 1
         )) AS duplicate_timestamps,

        (SELECT COUNT_IF(weather_datetime IS NULL)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS null_weather_datetime,

        (SELECT COUNT_IF(temperature_2m IS NULL)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS null_temperature_2m,

        (SELECT COUNT_IF(precipitation IS NULL)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS null_precipitation,

        (SELECT COUNT_IF(rain IS NULL)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS null_rain,

        (SELECT COUNT_IF(snowfall IS NULL)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS null_snowfall,

        (SELECT COUNT_IF(weather_code IS NULL)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS null_weather_code,

        (SELECT COUNT_IF(wind_speed_10m IS NULL)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS null_wind_speed_10m,

        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`weather`
         WHERE weather_datetime < '2026-03-01 00:00:00'
            OR weather_datetime > '2026-05-31 23:59:59') AS timestamps_outside_period,

        (SELECT COUNT(*)
         FROM (
             SELECT weather_datetime,
                    LEAD(weather_datetime) OVER (ORDER BY weather_datetime) AS next_datetime,
                    TIMESTAMPDIFF(HOUR, weather_datetime, LEAD(weather_datetime) OVER (ORDER BY weather_datetime)) AS hours_gap
             FROM `ftw-week-08`.`02_silver`.`weather`
         )
         WHERE hours_gap > 1) AS missing_hourly_gaps,

        (SELECT COUNT(*)
         FROM `ftw-week-08`.`01_bronze`.`weather_raw`
         WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
           AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json') AS approved_bronze_batch_count,

        (SELECT SIZE(from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>').hourly.time)
         FROM `ftw-week-08`.`01_bronze`.`weather_raw`
         WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
           AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json') AS json_hourly_count,

        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`weather`
         WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
           AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json') AS silver_row_count,

        (SELECT COUNT_IF(precipitation < 0)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS negative_precipitation,

        (SELECT COUNT_IF(rain < 0)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS negative_rain,

        (SELECT COUNT_IF(snowfall < 0)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS negative_snowfall,

        (SELECT COUNT_IF(wind_speed_10m < 0)
         FROM `ftw-week-08`.`02_silver`.`weather`) AS negative_wind_speed,

        (SELECT COUNT(*)
         FROM `ftw-week-08`.`02_silver`.`weather` w
         WHERE weather_code IS NOT NULL
           AND weather_code NOT IN (
             0, 1, 2, 3, 45, 48, 51, 53, 55, 56, 57,
             61, 63, 65, 66, 67, 71, 73, 75, 77,
             80, 81, 82, 85, 86, 95, 96, 99
           )) AS invalid_weather_codes,

        (SELECT 
           CASE
             WHEN json_parse_status = 'FAILED' THEN 1
             WHEN time_count = 0 THEN 1
             WHEN time_count = temperature_2m_count
              AND time_count = precipitation_count
              AND time_count = rain_count
              AND time_count = snowfall_count
              AND time_count = weather_code_count
              AND time_count = wind_speed_10m_count
             THEN 0
             ELSE 1
           END
         FROM (
           SELECT
             CASE WHEN json_data IS NOT NULL THEN 'SUCCESS' ELSE 'FAILED' END AS json_parse_status,
             SIZE(json_data.hourly.time) AS time_count,
             SIZE(json_data.hourly.temperature_2m) AS temperature_2m_count,
             SIZE(json_data.hourly.precipitation) AS precipitation_count,
             SIZE(json_data.hourly.rain) AS rain_count,
             SIZE(json_data.hourly.snowfall) AS snowfall_count,
             SIZE(json_data.hourly.weather_code) AS weather_code_count,
             SIZE(json_data.hourly.wind_speed_10m) AS wind_speed_10m_count
           FROM (
             SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, utc_offset_seconds INT, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
             FROM `ftw-week-08`.`01_bronze`.`weather_raw`
             WHERE source_file = 'open_meteo_2026-03-01_2026-05-31.json'
               AND batch_id = 'open_meteo_2026-03-01_2026-05-31.json'
           )
         )) AS array_alignment_failure
)

SELECT
    check_name,
    expected,
    actual,
    CASE
        WHEN expected = actual THEN 'PASS'
        ELSE 'FAIL'
    END AS status
FROM (
    SELECT 'Expected hourly observations' AS check_name, expected_observations AS expected, actual_observations AS actual
    FROM validation_results

    UNION ALL
    SELECT 'Duplicate timestamps', 0, duplicate_timestamps
    FROM validation_results

    UNION ALL
    SELECT 'NULL weather_datetime', 0, null_weather_datetime
    FROM validation_results

    UNION ALL
    SELECT 'NULL temperature_2m', 0, null_temperature_2m
    FROM validation_results

    UNION ALL
    SELECT 'NULL precipitation', 0, null_precipitation
    FROM validation_results

    UNION ALL
    SELECT 'NULL rain', 0, null_rain
    FROM validation_results

    UNION ALL
    SELECT 'NULL snowfall', 0, null_snowfall
    FROM validation_results

    UNION ALL
    SELECT 'NULL weather_code', 0, null_weather_code
    FROM validation_results

    UNION ALL
    SELECT 'NULL wind_speed_10m', 0, null_wind_speed_10m
    FROM validation_results

    UNION ALL
    SELECT 'Timestamps outside period', 0, timestamps_outside_period
    FROM validation_results

    UNION ALL
    SELECT 'Missing hourly gaps', 0, missing_hourly_gaps
    FROM validation_results

    UNION ALL
    SELECT 'Batch-specific JSON-to-Silver alignment', json_hourly_count, silver_row_count
    FROM validation_results

    UNION ALL
    SELECT 'Negative precipitation', 0, negative_precipitation
    FROM validation_results

    UNION ALL
    SELECT 'Negative rain', 0, negative_rain
    FROM validation_results

    UNION ALL
    SELECT 'Negative snowfall', 0, negative_snowfall
    FROM validation_results

    UNION ALL
    SELECT 'Negative wind_speed', 0, negative_wind_speed
    FROM validation_results

    UNION ALL
    SELECT 'Invalid WMO weather codes', 0, invalid_weather_codes
    FROM validation_results

    UNION ALL
    SELECT 'Invalid array alignment', 0, array_alignment_failure
    FROM validation_results

    UNION ALL
    SELECT 'Approved Bronze batch count', 1, approved_bronze_batch_count
    FROM validation_results
) checks
ORDER BY CASE check_name
    WHEN 'Expected hourly observations' THEN 1
    WHEN 'Duplicate timestamps' THEN 2
    WHEN 'NULL weather_datetime' THEN 3
    WHEN 'NULL temperature_2m' THEN 4
    WHEN 'NULL precipitation' THEN 5
    WHEN 'NULL rain' THEN 6
    WHEN 'NULL snowfall' THEN 7
    WHEN 'NULL weather_code' THEN 8
    WHEN 'NULL wind_speed_10m' THEN 9
    WHEN 'Timestamps outside period' THEN 10
    WHEN 'Missing hourly gaps' THEN 11
    WHEN 'Batch-specific JSON-to-Silver alignment' THEN 12
    WHEN 'Negative precipitation' THEN 13
    WHEN 'Negative rain' THEN 14
    WHEN 'Negative snowfall' THEN 15
    WHEN 'Negative wind_speed' THEN 16
    WHEN 'Invalid WMO weather codes' THEN 17
    WHEN 'Invalid array alignment' THEN 18
    WHEN 'Approved Bronze batch count' THEN 19
END

### Validation Summary

**Silver Weather Table: `ftw-week-08`.`02_silver`.`weather`**

**Final Validation Status:** See execution results above for comprehensive pass/fail status (19 checks)

The final validation summary above shows comprehensive pass/fail checks for:
* **Coverage & Completeness:** Expected hourly observations vs actual, no missing hourly gaps
* **Data Integrity:** Zero duplicates, no NULL values in any weather field
* **Timestamp Range:** All timestamps within 2026-03-01 to 2026-05-31 period
* **Bronze-to-Silver Alignment:** Batch-specific reconciliation of approved source batch
* **Data Quality:** No negative values in precipitation, rain, snowfall, or wind speed fields
* **WMO Weather Codes:** All weather_code values match documented Open-Meteo WMO codes
* **Array Alignment:** JSON array lengths match before POSEXPLODE transformation

**DST Inspection Finding (March 8, 2026):**
* The source provides 24 local-hour labels (00:00 through 23:00) for the DST transition day.
* This is an observation of the source data, not proof of DST correctness.

**Schema:**
* `weather_datetime` (TIMESTAMP_NTZ) - Hourly timestamp (NYC local-hour labels, no timezone conversion)
* `latitude` (DOUBLE) - Source location latitude
* `longitude` (DOUBLE) - Source location longitude
* `timezone` (STRING) - Source timezone identifier
* `timezone_abbreviation` (STRING) - Source timezone abbreviation
* `utc_offset_seconds` (INT) - UTC offset in seconds
* `temperature_2m` (DOUBLE) - Temperature at 2 meters in Celsius
* `precipitation` (DOUBLE) - Total precipitation in mm
* `rain` (DOUBLE) - Rain volume in mm
* `snowfall` (DOUBLE) - Snowfall volume in cm
* `weather_code` (INT) - WMO weather code
* `wind_speed_10m` (DOUBLE) - Wind speed at 10 meters in km/h
* Bronze provenance: `source_system`, `source_url`, `source_file`, `batch_id`, `ingested_at`

**Source Batch:** `open_meteo_2026-03-01_2026-05-31.json`
